<a href="https://colab.research.google.com/github/dr-mushtaq/Generative-AI-/blob/main/Chapter_3_Gen_AI_Foundational_Models_for_NLP_and_Language_Understanding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Document Classification with Torchtext (AG_NEWS)# New Section

In [2]:
# ============================================================
# AG News Text Classification — Colab-ready version
#
# WHY THIS WAS CHANGED FROM THE ORIGINAL:
# 1. torchtext.datasets.AG_NEWS and torchtext.vocab often break on
#    Google Colab because Colab ships a newer `torch` than the last
#    torchtext release supports (torchtext is no longer actively
#    maintained), and its download URLs are frequently blocked/broken.
#    -> Replaced with Hugging Face `datasets`, which is actively
#       maintained and installs cleanly on Colab.
# 2. torchtext's tokenizer/vocab utilities were dropped for the same
#    reason -> replaced with a small dependency-free tokenizer + vocab
#    built with `collections.Counter`.
# 3. The original script built and forward-passed the model but never
#    trained it, so all predictions were from random weights.
#    -> Added a real training loop (loss, optimizer, epochs, accuracy).
# 4. Added GPU/device handling since Colab usually provides a GPU.
#
# HOW TO USE IN COLAB:
# - Paste this whole file into a single Colab cell (or split at the
#   marked "NEW CELL" comments), then Runtime > Run all.
# - The first line installs `datasets`; if you split into cells, keep
#   that install line in its own cell and run it first.
# ============================================================

# NEW CELL -----------------------------------------------------
get_ipython().system('pip install -q datasets') if 'get_ipython' in globals() else None
# If the line above errors outside a notebook, just run this once
# in its own Colab cell instead:  !pip install -q datasets

import re
import torch
import torch.nn as nn
from collections import Counter
from torch.utils.data import DataLoader
from datasets import load_dataset

# ============================
# 1. Load Dataset (Hugging Face `datasets` instead of torchtext)
#    Note: the dataset now lives at "fancyzhx/ag_news" on the Hub —
#    the old bare "ag_news" id no longer resolves.
# ============================
raw_dataset = load_dataset("fancyzhx/ag_news")
train_data = raw_dataset["train"]
test_data = raw_dataset["test"]

# HF ag_news labels are already 0-indexed: 0=World,1=Sports,2=Business,3=Sci/Tech
ag_news_label = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

# View one sample
sample = train_data[0]
print("Sample Label :", ag_news_label[sample["label"]])
print("Sample Text  :", sample["text"])

# ============================
# 2. Tokenizer (simple, dependency-free replacement for
#    torchtext's get_tokenizer("basic_english"))
# ============================
_token_re = re.compile(r"[A-Za-z]+|\d+")

def tokenizer(text):
    return [tok.lower() for tok in _token_re.findall(text)]

# ============================
# 3. Build Vocabulary (replacement for build_vocab_from_iterator)
# ============================
def yield_tokens(dataset):
    for row in dataset:
        yield tokenizer(row["text"])

counter = Counter()
for tokens in yield_tokens(train_data):
    counter.update(tokens)

MIN_FREQ = 1
itos = ["<unk>"] + [tok for tok, freq in counter.items() if freq >= MIN_FREQ]
stoi = {tok: idx for idx, tok in enumerate(itos)}
UNK_INDEX = stoi["<unk>"]

def vocab(tokens):
    return [stoi.get(tok, UNK_INDEX) for tok in tokens]

print("\nVocabulary Size:", len(itos))

# ============================
# 4. Pipelines
# ============================
def text_pipeline(text):
    return vocab(tokenizer(text))

def label_pipeline(label):
    return int(label)  # already 0-indexed from HF dataset

# ============================
# 5. Collate Function
# ============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nUsing device:", device)

def collate_batch(batch):

    label_list = []
    text_list = []
    offsets = [0]

    for row in batch:

        label_list.append(label_pipeline(row["label"]))

        processed_text = torch.tensor(
            text_pipeline(row["text"]),
            dtype=torch.int64
        )

        text_list.append(processed_text)

        offsets.append(processed_text.size(0))

    label_list = torch.tensor(label_list, dtype=torch.int64)
    offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)
    text_list = torch.cat(text_list)

    return (
        label_list.to(device),
        text_list.to(device),
        offsets.to(device),
    )

# ============================
# 6. DataLoaders
# ============================
BATCH_SIZE = 64

train_dataloader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch
)

test_dataloader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)

labels, text, offsets = next(iter(train_dataloader))

print("\nBatch Labels Shape :", labels.shape)
print("Text Tensor Shape  :", text.shape)
print("Offsets Shape      :", offsets.shape)

# ============================
# 7. Model Definition
# ============================
class TextClassificationModel(nn.Module):

    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()

        self.embedding = nn.EmbeddingBag(
            vocab_size,
            embed_dim,
            sparse=False
        )

        self.fc = nn.Linear(embed_dim, num_class)

        self.init_weights()

    def init_weights(self):

        initrange = 0.5

        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.fc.weight.data.uniform_(-initrange, initrange)
        self.fc.bias.data.zero_()

    def forward(self, text, offsets):

        embedded = self.embedding(text, offsets)

        return self.fc(embedded)

# ============================
# 8. Create Model
# ============================
VOCAB_SIZE = len(itos)
EMBED_DIM = 64
NUM_CLASS = 4

model = TextClassificationModel(
    VOCAB_SIZE,
    EMBED_DIM,
    NUM_CLASS
).to(device)

# ============================
# 9. Training Loop (new — the original script never trained the model)
# ============================
EPOCHS = 5
LEARNING_RATE = 5.0

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 1, gamma=0.1)

def evaluate(dataloader):
    model.eval()
    total_acc, total_count = 0, 0
    with torch.no_grad():
        for labels, text, offsets in dataloader:
            output = model(text, offsets)
            total_acc += (output.argmax(1) == labels).sum().item()
            total_count += labels.size(0)
    return total_acc / total_count

def train_one_epoch(dataloader, epoch):
    model.train()
    total_acc, total_count = 0, 0
    log_interval = 200

    for idx, (labels, text, offsets) in enumerate(dataloader):

        optimizer.zero_grad()
        output = model(text, offsets)
        loss = criterion(output, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()

        total_acc += (output.argmax(1) == labels).sum().item()
        total_count += labels.size(0)

        if idx % log_interval == 0 and idx > 0:
            print(
                f"Epoch {epoch} | Batch {idx}/{len(dataloader)} "
                f"| Train Accuracy: {total_acc / total_count:.3f}"
            )
            total_acc, total_count = 0, 0

print("\nTraining...\n")
for epoch in range(1, EPOCHS + 1):
    train_one_epoch(train_dataloader, epoch)
    scheduler.step()
    test_acc = evaluate(test_dataloader)
    print(f"--- End of Epoch {epoch} | Test Accuracy: {test_acc:.3f} ---\n")

# ============================
# 10. Prediction Function
# ============================
def predict(text):

    model.eval()

    with torch.no_grad():

        processed_text = torch.tensor(
            text_pipeline(text),
            dtype=torch.int64
        ).to(device)

        output = model(
            processed_text,
            torch.tensor([0]).to(device)
        )

        return ag_news_label[output.argmax(1).item()]

# ============================
# 11. Test Predictions (now from a trained model)
# ============================
print("\nPrediction Examples (Trained Model)\n")

examples = [
    "Apple launches a new AI-powered iPhone.",
    "Pakistan won the cricket world cup.",
    "Stock markets gained after interest rates fell.",
    "Earthquake kills hundreds in Asia.",
]

for ex in examples:
    print("Text       :", ex)
    print("Prediction :", predict(ex))
    print()


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Sample Label : Business
Sample Text  : Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.

Vocabulary Size: 63230

Using device: cpu

Batch Labels Shape : torch.Size([64])
Text Tensor Shape  : torch.Size([2479])
Offsets Shape      : torch.Size([64])


[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.



Training...

Epoch 1 | Batch 200/1875 | Train Accuracy: 0.532
Epoch 1 | Batch 400/1875 | Train Accuracy: 0.787
Epoch 1 | Batch 600/1875 | Train Accuracy: 0.839
Epoch 1 | Batch 800/1875 | Train Accuracy: 0.854
Epoch 1 | Batch 1000/1875 | Train Accuracy: 0.873
Epoch 1 | Batch 1200/1875 | Train Accuracy: 0.876
Epoch 1 | Batch 1400/1875 | Train Accuracy: 0.881
Epoch 1 | Batch 1600/1875 | Train Accuracy: 0.884
Epoch 1 | Batch 1800/1875 | Train Accuracy: 0.889
--- End of Epoch 1 | Test Accuracy: 0.890 ---

Epoch 2 | Batch 200/1875 | Train Accuracy: 0.905
Epoch 2 | Batch 400/1875 | Train Accuracy: 0.909
Epoch 2 | Batch 600/1875 | Train Accuracy: 0.906
Epoch 2 | Batch 800/1875 | Train Accuracy: 0.906
Epoch 2 | Batch 1000/1875 | Train Accuracy: 0.906
Epoch 2 | Batch 1200/1875 | Train Accuracy: 0.904
Epoch 2 | Batch 1400/1875 | Train Accuracy: 0.903
Epoch 2 | Batch 1600/1875 | Train Accuracy: 0.904
Epoch 2 | Batch 1800/1875 | Train Accuracy: 0.904
--- End of Epoch 2 | Test Accuracy: 0.893 ---



In [3]:
# ============================
# 11. Test Predictions (now from a trained model)
# ============================
print("\nPrediction Examples (Trained Model)\n")

examples = [
    "Apple launches a new AI-powered iPhone.",
    "Pakistan won the cricket world cup.",
    "Stock markets gained after interest rates fell.",
    "Earthquake kills hundreds in Asia.",
]

for ex in examples:
    print("Text       :", ex)
    print("Prediction :", predict(ex))
    print()



Prediction Examples (Trained Model)

Text       : Apple launches a new AI-powered iPhone.
Prediction : Sci/Tech

Text       : Pakistan won the cricket world cup.
Prediction : Sports

Text       : Stock markets gained after interest rates fell.
Prediction : Business

Text       : Earthquake kills hundreds in Asia.
Prediction : World

